In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [26]:
from pathlib import Path
import librosa
import soundfile as sf
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# PATHS

BASE_DIR = Path(
    "/content/drive/Shareddrives/Safe Cue Project/SafeCue Audio Recordings"
)

INPUT_DIR = BASE_DIR / "split_data"
OUTPUT_DIR = BASE_DIR / "one_second_clips"

# SETTINGS

SAMPLE_RATE = 16000
CLIP_LENGTH = 1  # seconds
SAMPLES_PER_CLIP = SAMPLE_RATE * CLIP_LENGTH

# FUNCTION TO SLICE ONE AUDIO FILE


def slice_audio(input_file, output_folder):

    # Load audio at 16 kHz and convert to mono
    audio, sr = librosa.load(
        input_file,
        sr=SAMPLE_RATE,
        mono=True
    )

    # Calculate how many 1-second clips we need
    num_clips = (len(audio) + SAMPLES_PER_CLIP - 1) // SAMPLES_PER_CLIP

    # Create output folder
    output_folder.mkdir(parents=True, exist_ok=True)

    # Create each 1-second clip
    for i in range(num_clips):

        start = i * SAMPLES_PER_CLIP
        end = start + SAMPLES_PER_CLIP

        clip = audio[start:end]

        # If final clip is shorter than 1 second,
        # pad it with zeros
        if len(clip) < SAMPLES_PER_CLIP:
            clip = librosa.util.fix_length(
                clip,
                size=SAMPLES_PER_CLIP
            )

        # Example:
        # original.wav -> original_0000.wav
        output_name = f"{input_file.stem}_{i:04d}.wav"

        output_file = output_folder / output_name

        # Save as WAV
        sf.write(
            output_file,
            clip,
            SAMPLE_RATE
        )


# PROCESS TRAIN / VALIDATION / TEST


splits = [
    "train",
    "validation",
    "test"
]

for split in splits:

    split_input = INPUT_DIR / split
    split_output = OUTPUT_DIR / split

    if not split_input.exists():
        print(f"Skipping {split}: folder not found")
        continue

    print(f"\nProcessing {split}...")

    # Go through each class
    for class_folder in split_input.iterdir():

        if not class_folder.is_dir():
            continue

        class_name = class_folder.name

        output_class_folder = split_output / class_name

        print(f"\n  {class_name}")

        # Find WAV files
        wav_files = list(class_folder.glob("*.wav"))

        print(f"    Found {len(wav_files)} original audio files")

        # Slice each audio file
        for audio_file in wav_files:

            slice_audio(
                audio_file,
                output_class_folder
            )

        # Count resulting clips
        clips_created = len(
            list(output_class_folder.glob("*.wav"))
        )

        print(f"    Created {clips_created} one-second clips")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Processing train...

  Fire Alarm
    Found 35 original audio files
    Created 512 one-second clips

  Smoke Alarm
    Found 35 original audio files
    Created 525 one-second clips

  Siren
    Found 35 original audio files
    Created 569 one-second clips

  Screaming
    Found 35 original audio files
    Created 478 one-second clips

  Background Noise
    Found 35 original audio files
    Created 551 one-second clips

Processing validation...

  Fire Alarm
    Found 8 original audio files
    Created 131 one-second clips

  Smoke Alarm
    Found 8 original audio files
    Created 122 one-second clips

  Siren
    Found 8 original audio files
    Created 131 one-second clips

  Screaming
    Found 8 original audio files
    Created 109 one-second clips

  Background Noise
    Found 8 original audio files
    Created 113 one-second clips

Processing test.